# 112 — Synthetic Training Data Generation for LLM Fine-Tuning
## What you'll learn: how to build a data flywheel that generates, filters, and validates labeled examples at scale
⏱ ~60 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/112-synthetic-training-data/synthetic_training_data_workbook.ipynb)

Fine-tuning a language model on your specific task dramatically improves accuracy and reduces inference cost. The bottleneck is **labeled data** — collecting thousands of human-written examples is slow and expensive. This workshop shows you how to generate that data synthetically, clean it rigorously, and export it ready for OpenAI fine-tuning.

Inspired by **Stanford Alpaca** (Wang et al. 2022 Self-Instruct) and **WizardLM** (Xu et al. 2023), which demonstrated that GPT-generated training data can match or exceed human-curated datasets for many tasks — when paired with deduplication and quality validation.

The task: **customer service tone classification** — labeling messages as `formal`, `informal`, or `urgent`. The same three-stage pipeline generalises to any classification or generation task.

---
### Workshop Roadmap
| # | Topic |
|---|-------|
| 1 | **Concepts** — data flywheel, why synthetic data works, pipeline overview |
| 2 | **Setup** — install deps, configure API key |
| 3 | **Stage 1: Generator** — prompting GPT-4o for diverse labeled examples |
| 4 | **Stage 2: Deduplication** — embedding similarity and the 0.95 threshold |
| 5 | **Stage 3: Judge validation** — second LLM for quality gating |
| 6 | **Full pipeline run** — end-to-end with JSONL inspection |
| 7 | **Cost analysis and scaling** — how far can this go? |
| ★ | **Exercises + Answer Key** |

---
### Prerequisites
- Python 3.10+, or Google Colab
- `OPENAI_API_KEY` in `.env` or Colab Secrets
- `langchain-openai`, `python-dotenv`, `numpy`

### Key References
> Wang et al. (2022) — *Self-Instruct: Aligning Language Models with Self-Generated Instructions* — [arxiv:2212.10560](https://arxiv.org/abs/2212.10560)
>
> Xu et al. (2023) — *WizardLM: Empowering Large Language Models to Follow Complex Instructions* — [arxiv:2304.12244](https://arxiv.org/abs/2304.12244)
>
> Lee et al. (2022) — *Deduplicating Training Data Makes Language Models Better* — [arxiv:2107.06499](https://arxiv.org/abs/2107.06499)
>
> [OpenAI Fine-tuning guide](https://platform.openai.com/docs/guides/fine-tuning)

## Part 1 — Concepts

### The data flywheel

The core insight of Self-Instruct and WizardLM is that a capable LLM can bootstrap its own training data:

```
┌─────────────────────────────────────────────────────────────────┐
│                    The Synthetic Data Flywheel                  │
│                                                                 │
│   Strong LLM (GPT-4o)                                           │
│        │                                                        │
│        │  generates                                             │
│        ▼                                                        │
│   Raw examples  ──► deduplicate ──► judge ──► JSONL            │
│                                               │                 │
│                                               │ fine-tune on    │
│                                               ▼                 │
│                                         Smaller LLM            │
│                                    (cheaper at inference)       │
└─────────────────────────────────────────────────────────────────┘
```

You spend generation tokens once (at data creation time) and save inference tokens forever (at serving time). A fine-tuned `gpt-4o-mini` on a specific task often outperforms the base `gpt-4o` on that task while costing ~50x less per call.

### Why synthetic data works

Three reasons GPT-generated examples rival human ones for classification tasks:

1. **Label consistency**: a human-in-a-loop labeling operation drifts as annotators fatigue; a prompted LLM applies the same definition of "urgent" on example 1 and example 1000.
2. **Volume**: generating 1000 examples costs ~$0.10 with `gpt-4o-mini`; human labeling at $0.10/example costs $100.
3. **Diversity control**: you can explicitly vary industry, phrasing, and length through your generation prompt — something that's hard to guarantee with crowdsourced data.

The risk: **mode collapse** — the generator samples similar examples repeatedly. This is what the deduplication stage catches.

### The three pipeline stages

```
Stage 1  GENERATE   GPT-4o produces raw labeled messages
    │
    ▼
Stage 2  DEDUPLICATE  Embedding similarity removes near-duplicates (sim > 0.95)
    │
    ▼
Stage 3  JUDGE       Second LLM validates label correctness and message clarity
    │
    ▼
         EXPORT      JSONL file in OpenAI fine-tuning format
```

Each stage has a clear success metric: generation volume, deduplication rate, and judge acceptance rate. Together they produce a high-quality, diverse, validated dataset.

## Part 2 — Setup

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "langchain-openai", "langchain-core", "python-dotenv", "numpy"],
        check=True
    )
    print("Colab install complete.")
else:
    print("Local — skipping install (using requirements.txt)")

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
print(f"API key ready: {bool(key) and key.startswith('sk-')}")

## Part 3 — Stage 1: Generator

### What the generator does

`generate_examples_for_label(label, n, llm)` sends a single LLM call that asks GPT-4o to produce `n` diverse customer service messages with the requested tone. It:

1. Uses a **system prompt** (`GENERATOR_SYSTEM`) that instructs the model to vary industry, length, and phrasing
2. Embeds the label name and count in the **user prompt**
3. Parses the numbered-list response into individual message strings

```
GENERATOR_SYSTEM ─► "You are a dataset generator creating diverse, realistic
                      customer service messages for a tone classification
                      training set. Vary phrasing, length, context, and
                      industry while clearly belonging to the requested
                      tone class."

User prompt ─► "Generate 5 diverse customer service messages that have a
                clearly 'formal' tone. Number each: 1. ... 2. ..."

Response ─► 1. "Dear Support Team, I am writing to formally request..."
            2. "I wish to bring to your attention that my recent order..."
            3. ...
```

### Chain-of-thought variation

The generator prompt includes three structural variation axes:
- **Industry**: e-commerce, banking, healthcare, travel, SaaS
- **Length**: 1 sentence vs. 2-3 sentences
- **Phrasing**: polite request vs. complaint vs. inquiry

Specifying these variations in the prompt is equivalent to chain-of-thought for diversity — the model generates from a wider part of its distribution rather than clustering around the most common pattern.

### The three labels

| Label | Semantic definition |
|-------|---------------------|
| `formal` | Professional register, complete sentences, no contractions |
| `informal` | Casual tone, contractions, colloquialisms, possibly abbreviations |
| `urgent` | Time pressure, urgency markers ("ASAP", "immediately"), emotional stakes |

In [ ]:
import json
from dataclasses import dataclass, field

import numpy as np
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import SystemMessage, HumanMessage


LABELS = ["formal", "informal", "urgent"]

GENERATOR_SYSTEM = (
    "You are a dataset generator creating diverse, realistic customer service messages "
    "for a tone classification training set. "
    "Generate messages that vary in phrasing, length, context, and industry "
    "while clearly belonging to the requested tone class. "
    "Each message should be 1-3 sentences. Be creative and avoid repetition."
)

JUDGE_SYSTEM = (
    "You are a dataset quality judge. "
    "Given a customer service message and its assigned label, decide if the label is correct "
    "and the message is clear. "
    'Reply with JSON only: {"valid": true/false, "reason": "one sentence"}.'
)


@dataclass
class GeneratedExample:
    message: str
    label: str
    is_valid: bool = True
    judge_reason: str = ""
    embedding: list[float] = field(default_factory=list)


print("Imports and data classes ready.")

In [ ]:
def generate_examples_for_label(label: str, n: int, llm: ChatOpenAI) -> list[str]:
    """
    Use GPT-4o to generate N diverse examples for a given label.
    Varies industry and phrasing to maximise diversity.
    """
    prompt = (
        f"Generate {n} diverse customer service messages that have a clearly '{label}' tone.\n"
        "Requirements:\n"
        "- Each message should be on its own line\n"
        "- Vary the industry, product type, and phrasing\n"
        "- Include a mix of short (1 sentence) and medium (2-3 sentence) messages\n"
        "- Make each message distinctly different from the others\n"
        f"- Number each message: 1. ... 2. ...\n\n"
        f"Generate {n} messages now:"
    )
    response = llm.invoke([
        SystemMessage(content=GENERATOR_SYSTEM),
        HumanMessage(content=prompt),
    ])
    raw = response.content.strip()

    examples = []
    for line in raw.split("\n"):
        line = line.strip()
        if not line:
            continue
        for i in range(1, n + 10):
            prefix = f"{i}. "
            if line.startswith(prefix):
                line = line[len(prefix):].strip()
                break
        if len(line) > 10:
            examples.append(line)

    return examples[:n]


print("generate_examples_for_label() defined.")

In [ ]:
# Generate 3 examples per label — small batch for demonstration

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.9)

print("Generating examples...\n")

for label in LABELS:
    messages = generate_examples_for_label(label, n=3, llm=llm)
    print(f"Label: {label.upper()}")
    for i, msg in enumerate(messages, 1):
        print(f"  {i}. {msg}")
    print()

### Why temperature=0.9?

Generation uses `temperature=0.9` (vs. `temperature=0` in the judge). Here is why:

```
temperature=0    → deterministic, most likely token at each step
                   Good for: judgment, extraction, structured output
                   Risk for generation: near-identical outputs every run

temperature=0.9  → high diversity, more sampling from the full distribution
                   Good for: data generation, creative variation
                   Risk: occasional incoherent outputs (the judge catches these)
```

The pipeline deliberately separates concerns: **diversity** is the generator's job; **accuracy** is the judge's job. Each stage uses the temperature setting appropriate for its role.

### Inspecting the raw output format

The generator returns a numbered list that the parser strips. Let's see the raw response before parsing to understand what we're working with:

In [ ]:
# Show the raw response before parsing — for learning purposes only

raw_prompt = (
    "Generate 4 diverse customer service messages that have a clearly 'urgent' tone.\n"
    "Requirements:\n"
    "- Each message should be on its own line\n"
    "- Vary the industry, product type, and phrasing\n"
    "- Include a mix of short (1 sentence) and medium (2-3 sentence) messages\n"
    "- Make each message distinctly different from the others\n"
    "- Number each message: 1. ... 2. ...\n\n"
    "Generate 4 messages now:"
)

raw_response = llm.invoke([
    SystemMessage(content=GENERATOR_SYSTEM),
    HumanMessage(content=raw_prompt),
])

print("Raw model response:")
print("---")
print(raw_response.content)
print("---")
print(f"\nTotal characters: {len(raw_response.content)}")

## Part 4 — Stage 2: Deduplication

### The deduplication problem

When you generate hundreds of examples, the LLM repeats itself. Not with identical text (which would be trivial to catch with exact-match deduplication) but with **semantically similar** text:

```
Example A: "I need this issue resolved immediately — it's affecting my business."
Example B: "This needs to be fixed right away as it is seriously impacting my operations."
```

These are different strings but near-identical semantics. A fine-tuned model trained on both will overfit to this pattern.

Lee et al. (2022) showed that deduplication improves both perplexity and downstream task performance — and that **semantic** deduplication (embedding-based) outperforms exact-match or n-gram deduplication.

### The embedding pipeline

```
texts ─► OpenAI text-embedding-3-small ─► 1536-dimensional float vectors

embed_examples(["msg1", "msg2", ...]) ─► [[0.02, -0.14, ...], [0.09, 0.03, ...]]
```

`text-embedding-3-small` is the standard choice here: it costs $0.00002/1K tokens (50x cheaper than GPT-4o input) and produces high-quality semantic representations for short text.

### Cosine similarity

For two vectors **a** and **b**:

```
                    a · b
cos_sim(a, b) = ──────────────
                ‖a‖ × ‖b‖

Range: [-1, 1]
  1.0  → identical direction (near-duplicate)
  0.0  → orthogonal (unrelated)
 -1.0  → opposite direction (antonyms at extremes)
```

For sentence embeddings, values above 0.9 typically indicate paraphrase-level similarity.

### Why threshold=0.95?

The 0.95 threshold is deliberately conservative:

| Threshold | Behaviour |
|-----------|-----------|
| 0.99 | Only catches near-exact copies; most paraphrases survive |
| 0.95 | Removes clear paraphrases; keeps stylistic variation |
| 0.90 | Aggressively removes similar examples; may over-filter |
| 0.80 | Removes same-topic examples regardless of phrasing |

At 0.95, two examples must be semantically almost identical to be dropped. This preserves genuine variation in how "formal" can be expressed while eliminating redundant repetition.

In [ ]:
def embed_examples(texts: list[str], embeddings: OpenAIEmbeddings) -> list[list[float]]:
    """Embed a list of texts using OpenAI text-embedding-3-small."""
    return embeddings.embed_documents(texts)


def cosine_similarity(a: list[float], b: list[float]) -> float:
    """Compute cosine similarity between two embedding vectors."""
    a_arr = np.array(a, dtype=float)
    b_arr = np.array(b, dtype=float)
    denom = np.linalg.norm(a_arr) * np.linalg.norm(b_arr)
    if denom == 0:
        return 0.0
    return float(np.dot(a_arr, b_arr) / denom)


def deduplicate_by_embedding(
    examples: list[GeneratedExample],
    threshold: float = 0.95,
) -> tuple[list[GeneratedExample], int]:
    """
    Remove near-duplicate examples using cosine similarity.
    Any pair with similarity > threshold → drop the later one.
    Returns (deduplicated_examples, n_dropped).
    """
    kept: list[GeneratedExample] = []
    dropped = 0
    for candidate in examples:
        if not candidate.embedding:
            kept.append(candidate)
            continue
        too_similar = any(
            cosine_similarity(candidate.embedding, existing.embedding) > threshold
            for existing in kept
            if existing.embedding
        )
        if too_similar:
            dropped += 1
        else:
            kept.append(candidate)
    return kept, dropped


print("embed_examples(), cosine_similarity(), deduplicate_by_embedding() defined.")

In [ ]:
# Demonstrate cosine similarity with hand-crafted examples

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

test_texts = [
    "I need this resolved immediately — it's impacting my business.",   # urgent A
    "This needs to be fixed right away as it is affecting my operations.",  # urgent B (near-dup)
    "Dear Support, I am writing to formally request a refund.",          # formal
]

test_vectors = embed_examples(test_texts, embedding_model)

print("Cosine similarity matrix:")
print(f"{'':45} A      B      C")
for i, (text_i, vec_i) in enumerate(zip(test_texts, test_vectors)):
    label = chr(65 + i)
    row = [cosine_similarity(vec_i, test_vectors[j]) for j in range(len(test_vectors))]
    row_str = "  ".join(f"{v:.3f}" for v in row)
    print(f"  {label}. {text_i[:42]:<42} {row_str}")

print("\nObservation:")
print(f"  A vs B (paraphrases): {cosine_similarity(test_vectors[0], test_vectors[1]):.3f}")
print(f"  A vs C (different):   {cosine_similarity(test_vectors[0], test_vectors[2]):.3f}")
print("\n  A vs B is above 0.95 threshold → B would be dropped.")

In [ ]:
# Generate a larger batch and run deduplication

print("Generating 5 examples per label (15 total)...")
llm_gen = ChatOpenAI(model="gpt-4o-mini", temperature=0.9)

raw_examples: list[GeneratedExample] = []
for label in LABELS:
    messages = generate_examples_for_label(label, n=5, llm=llm_gen)
    for msg in messages:
        raw_examples.append(GeneratedExample(message=msg, label=label))

print(f"Generated {len(raw_examples)} examples")

# Embed all at once (one API call — cheaper than embedding one by one)
all_texts = [ex.message for ex in raw_examples]
all_vectors = embed_examples(all_texts, embedding_model)

for ex, vec in zip(raw_examples, all_vectors):
    ex.embedding = vec

print(f"Embedded {len(all_vectors)} examples")

# Run deduplication
deduped, n_dropped = deduplicate_by_embedding(raw_examples, threshold=0.95)

print(f"\nDeduplication results:")
print(f"  Before: {len(raw_examples)}")
print(f"  After:  {len(deduped)}")
print(f"  Dropped: {n_dropped} ({n_dropped / len(raw_examples) * 100:.0f}% dedup rate)")

## Part 5 — Stage 3: Judge Validation

### Why a second LLM improves quality

The generator is optimised for diversity — it produces varied, creative messages. But diversity without accuracy is noise. Two failure modes the judge catches:

1. **Label drift**: the generator labels a message "formal" but the message contains slang or urgency markers. This is more common at high temperature.
2. **Ambiguous examples**: a message like "Please let me know when this is ready." could be formal or informal depending on context. The judge flags it as ambiguous, and these are excluded from training.

Training a classifier on ambiguous examples is harmful — the model learns inconsistent signal that reduces its calibration.

### The judge prompt design

```
JUDGE_SYSTEM: "You are a dataset quality judge.
               Given a customer service message and its assigned label,
               decide if the label is correct and the message is clear.
               Reply with JSON only: {"valid": true/false, "reason": "..."}"

User message: "Message: [message text]
               Label: [label]
               Is this label correct and is the message unambiguous?"
```

Requesting **JSON-only** output makes the response easy to parse reliably. Requiring a `reason` field serves two purposes: it forces the model to articulate its judgment (improving accuracy) and produces a human-readable audit trail.

### Judge at temperature=0

The judge uses `temperature=0` — deterministic output. We want the same judgment every time for the same example. Non-determinism in a judge would mean some examples randomly pass or fail on re-runs, making the dataset non-reproducible.

In [ ]:
def judge_example(example: GeneratedExample, llm: ChatOpenAI) -> GeneratedExample:
    """
    Second LLM call to validate label correctness and message clarity.
    Updates example.is_valid and example.judge_reason in place.
    """
    prompt = (
        f"Message: {example.message}\n"
        f"Label: {example.label}\n\n"
        "Is this label correct and is the message unambiguous? "
        'Reply with JSON only: {"valid": true/false, "reason": "one sentence"}'
    )
    response = llm.invoke([
        SystemMessage(content=JUDGE_SYSTEM),
        HumanMessage(content=prompt),
    ])
    try:
        data = json.loads(response.content.strip())
        example.is_valid = bool(data.get("valid", True))
        example.judge_reason = data.get("reason", "")
    except (json.JSONDecodeError, KeyError):
        example.is_valid = True
        example.judge_reason = "judge parse error — kept"
    return example


print("judge_example() defined.")

In [ ]:
# Run the judge over all deduplicated examples

llm_judge = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print(f"Judging {len(deduped)} examples...\n")
judged = []
for ex in deduped:
    judged_ex = judge_example(ex, llm_judge)
    judged.append(judged_ex)

# Summary
valid_count = sum(1 for ex in judged if ex.is_valid)
invalid_count = len(judged) - valid_count

print(f"Judge results:")
print(f"  Total judged: {len(judged)}")
print(f"  Valid:        {valid_count}")
print(f"  Invalid:      {invalid_count} ({invalid_count / len(judged) * 100:.0f}% rejection rate)")

In [ ]:
# Inspect the judge decisions — especially any invalid ones

print("Judge decisions per example:\n")
print(f"{'#':<4} {'Label':<10} {'Valid':<8} {'Reason'}")
print("-" * 80)
for i, ex in enumerate(judged, 1):
    valid_str = "YES" if ex.is_valid else "NO"
    reason_preview = ex.judge_reason[:55] if ex.judge_reason else "(no reason)"
    msg_preview = ex.message[:30]
    print(f"{i:<4} {ex.label:<10} {valid_str:<8} {reason_preview}")
    if not ex.is_valid:
        print(f"     Message: {msg_preview}...")

## Part 6 — JSONL Export

### The OpenAI fine-tuning format

OpenAI fine-tuning expects JSONL (one JSON object per line) where each object is a `messages` array in chat format:

```json
{
  "messages": [
    {"role": "system",    "content": "You are a tone classifier..."},
    {"role": "user",      "content": "I need this fixed immediately!"},
    {"role": "assistant", "content": "urgent"}
  ]
}
```

The system message defines the task. The user message is the input. The assistant message is the gold label. The model learns to map (system + user) → assistant.

### What `export_to_jsonl` does

- Iterates over all examples, skipping those where `is_valid=False`
- Constructs the chat-format dict for each
- Writes one JSON line per example
- Returns the count of exported examples

This count is the final metric for the pipeline: how many valid, deduplicated, judge-approved examples did we produce?

In [ ]:
def export_to_jsonl(examples: list[GeneratedExample], path: str) -> int:
    """
    Export valid examples as JSONL in OpenAI fine-tuning format.
    Returns count of exported examples.
    """
    system_msg = (
        "You are a tone classifier for customer service messages. "
        "Classify each message as exactly one of: formal, informal, urgent. "
        "Reply with only the label — no explanation."
    )
    count = 0
    with open(path, "w") as f:
        for ex in examples:
            if not ex.is_valid:
                continue
            row = {
                "messages": [
                    {"role": "system", "content": system_msg},
                    {"role": "user", "content": ex.message},
                    {"role": "assistant", "content": ex.label},
                ]
            }
            f.write(json.dumps(row) + "\n")
            count += 1
    return count


print("export_to_jsonl() defined.")

In [ ]:
# Export to JSONL and inspect the file

output_path = "/tmp/customer_tone_training.jsonl"
exported_count = export_to_jsonl(judged, output_path)

print(f"Exported {exported_count} examples to {output_path}")

# Read back and display first 3 lines
print("\nFirst 3 lines of the JSONL file:")
print("-" * 70)
with open(output_path) as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        row = json.loads(line)
        print(f"\nLine {i+1}:")
        for msg in row["messages"]:
            role = msg["role"].upper()
            content_preview = msg["content"][:60]
            print(f"  [{role}] {content_preview}")

## Part 7 — Full Pipeline Run

Now we run the complete three-stage pipeline end-to-end, mirroring what `main.py` does. This is the function you'd call in production to generate a training batch.

In [ ]:
# ===== Full pipeline — mirrors main.py =====

def run_pipeline(
    labels: list[str],
    n_per_label: int,
    output_path: str,
    dedup_threshold: float = 0.95,
) -> dict:
    """
    Full three-stage synthetic data pipeline.

    Stage 1: Generate n_per_label examples for each label
    Stage 2: Embed + deduplicate by cosine similarity
    Stage 3: Judge each remaining example for label accuracy
    Export:  Write valid examples to output_path as JSONL

    Returns a summary dict with counts at each stage.
    """
    gen_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.9)
    judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    emb_model = OpenAIEmbeddings(model="text-embedding-3-small")

    # Stage 1: Generate
    print(f"Stage 1  Generating {n_per_label} examples x {len(labels)} labels...")
    raw: list[GeneratedExample] = []
    for label in labels:
        texts = generate_examples_for_label(label, n=n_per_label, llm=gen_llm)
        for t in texts:
            raw.append(GeneratedExample(message=t, label=label))
    print(f"         Generated: {len(raw)}")

    # Stage 2: Embed + deduplicate
    print(f"Stage 2  Embedding {len(raw)} examples...")
    all_texts = [ex.message for ex in raw]
    vectors = embed_examples(all_texts, emb_model)
    for ex, vec in zip(raw, vectors):
        ex.embedding = vec

    deduped, n_dropped = deduplicate_by_embedding(raw, threshold=dedup_threshold)
    print(f"         Dropped {n_dropped} near-duplicates → {len(deduped)} remain")

    # Stage 3: Judge
    print(f"Stage 3  Judging {len(deduped)} examples...")
    judged_all = [judge_example(ex, judge_llm) for ex in deduped]
    n_invalid = sum(1 for ex in judged_all if not ex.is_valid)
    print(f"         Rejected {n_invalid} invalid examples → {len(judged_all) - n_invalid} valid")

    # Export
    count = export_to_jsonl(judged_all, output_path)
    print(f"Export   Wrote {count} examples to {output_path}")

    return {
        "generated": len(raw),
        "after_dedup": len(deduped),
        "n_dropped_dedup": n_dropped,
        "after_judge": len(judged_all),
        "n_invalid_judge": n_invalid,
        "exported": count,
        "dedup_rate": n_dropped / len(raw) if raw else 0,
        "rejection_rate": n_invalid / len(deduped) if deduped else 0,
    }


# Run a small full-pipeline demo (5 per label = 15 examples total)
summary = run_pipeline(
    labels=LABELS,
    n_per_label=5,
    output_path="/tmp/full_pipeline_output.jsonl",
    dedup_threshold=0.95,
)

print("\n=== Pipeline Summary ===")
for key, val in summary.items():
    if isinstance(val, float):
        print(f"  {key:<22} {val:.1%}")
    else:
        print(f"  {key:<22} {val}")

In [ ]:
# Inspect the final JSONL file — label distribution check

from collections import Counter

label_counts: Counter = Counter()
with open("/tmp/full_pipeline_output.jsonl") as f:
    for line in f:
        row = json.loads(line)
        # The assistant turn (last message) holds the label
        label = row["messages"][-1]["content"]
        label_counts[label] += 1

total = sum(label_counts.values())
print("Label distribution in exported JSONL:")
for label, count in sorted(label_counts.items()):
    bar = "=" * count
    pct = count / total * 100 if total else 0
    print(f"  {label:<10} {count:>3}  [{bar:<20}]  {pct:.0f}%")

print(f"\nTotal examples: {total}")
print("\nIdeal: roughly equal distribution across labels.")
print("Imbalance means the fine-tuned model will be biased toward the majority class.")

## Part 8 — Cost Analysis and Scaling

### Token counts for this pipeline

For `n_per_label=100` (300 examples total, a reasonable training dataset):

| Stage | Model | Estimated tokens | Estimated cost |
|-------|-------|-----------------|----------------|
| Generation (3 labels x 100) | gpt-4o-mini | ~45,000 | ~$0.007 |
| Embedding (300 examples) | text-embedding-3-small | ~15,000 | ~$0.0003 |
| Judgment (300 examples) | gpt-4o-mini | ~30,000 | ~$0.005 |
| **Total** | | **~90,000** | **~$0.012** |

300 training examples for $0.01. Compare to crowdsourced labeling at $0.10-0.50/example = $30-150 for the same volume.

### Scaling considerations

**How many examples do you need?**

OpenAI recommends at least 50-100 examples per class for fine-tuning, with diminishing returns above ~1,000. For a 3-class problem:
- Minimum viable: 50 x 3 = 150 examples (~$0.006)
- Solid baseline: 200 x 3 = 600 examples (~$0.025)
- Production-grade: 500 x 3 = 1,500 examples (~$0.060)

**The fine-tuning cost**

Training cost at OpenAI (gpt-4o-mini fine-tuning): ~$3.00 per 1M training tokens. A 600-example dataset with average 50 tokens/example = 30,000 tokens × 3 epochs = 90,000 tokens → ~$0.27 training cost. Total pipeline + training: under $1.

**Quality vs. quantity trade-off**

The dedup + judge stages reduce dataset size but increase quality. Empirically, 300 high-quality examples often outperform 1,000 noisy ones. The pipeline's funnel design is deliberate: generate at scale, then filter aggressively.

**Iterating on failures**

After fine-tuning, run inference on a held-out test set and inspect misclassified examples. Common patterns in errors → add generation prompts targeting those patterns → regenerate → re-fine-tune. This is the flywheel.

### Parallelising generation

For large batches, generate each label in parallel using `asyncio.gather` or `concurrent.futures.ThreadPoolExecutor`. The LangChain `ChatOpenAI` client supports async invocation natively:

In [ ]:
# Demonstrate async parallel generation — faster for large batches

import asyncio
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage


async def generate_label_async(label: str, n: int, llm: ChatOpenAI) -> tuple[str, list[str]]:
    """Async wrapper: returns (label, messages)."""
    prompt = (
        f"Generate {n} diverse customer service messages that have a clearly '{label}' tone.\n"
        "Requirements:\n"
        "- Each message on its own line\n"
        "- Vary the industry and phrasing\n"
        f"- Number each: 1. ... 2. ...\n\n"
        f"Generate {n} messages now:"
    )
    response = await llm.ainvoke([
        SystemMessage(content=GENERATOR_SYSTEM),
        HumanMessage(content=prompt),
    ])
    raw = response.content.strip()
    examples = []
    for line in raw.split("\n"):
        line = line.strip()
        if not line:
            continue
        for i in range(1, n + 10):
            if line.startswith(f"{i}. "):
                line = line[len(f"{i}. "):].strip()
                break
        if len(line) > 10:
            examples.append(line)
    return label, examples[:n]


async def generate_all_parallel(labels: list[str], n: int, llm: ChatOpenAI) -> list[GeneratedExample]:
    """Generate all labels in parallel — 3x faster than sequential."""
    tasks = [generate_label_async(label, n, llm) for label in labels]
    results = await asyncio.gather(*tasks)
    examples = []
    for label, messages in results:
        for msg in messages:
            examples.append(GeneratedExample(message=msg, label=label))
    return examples


# Run async parallel generation (3 labels simultaneously)
print("Generating 3 examples per label in parallel...")
async_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.9)

import time
start = time.time()
parallel_examples = await generate_all_parallel(LABELS, n=3, llm=async_llm)
elapsed = time.time() - start

print(f"Generated {len(parallel_examples)} examples in {elapsed:.1f}s")
for ex in parallel_examples:
    print(f"  [{ex.label}] {ex.message[:65]}")

## Exercises

### Exercise 1 — Add a fourth label: `technical`

The current pipeline generates three labels: `formal`, `informal`, `urgent`. Add a fourth label `technical` (engineering support tickets with domain-specific jargon) and regenerate a batch of 5 examples. Verify that the generator produces distinct, correctly-labelled technical examples.

Hint: add `"technical"` to the `LABELS` list and update `GENERATOR_SYSTEM` to include a definition.

---

### Exercise 2 — Tune the deduplication threshold

Run the deduplication stage twice on the same 15-example batch: once with `threshold=0.85` and once with `threshold=0.99`. Compare the number of examples dropped at each threshold. At what threshold do you start seeing obviously distinct examples being removed?

---

### Exercise 3 — Build a validation set splitter

A fine-tuning run needs a training set and a separate validation set. Implement `split_train_val(examples, val_fraction=0.2)` that:
1. Takes a list of `GeneratedExample` objects
2. Splits them into train and val, **stratified by label** (equal proportions of each label in both splits)
3. Returns `(train_examples, val_examples)`

Then export both splits to separate JSONL files.

In [ ]:
# ===== ANSWER KEY — Exercise 1: Add a fourth label =====

import random

# Add technical label with updated system prompt
LABELS_EXTENDED = ["formal", "informal", "urgent", "technical"]

GENERATOR_SYSTEM_V2 = (
    "You are a dataset generator creating diverse, realistic customer service messages "
    "for a tone classification training set. "
    "Generate messages that vary in phrasing, length, context, and industry "
    "while clearly belonging to the requested tone class. "
    "Each message should be 1-3 sentences. Be creative and avoid repetition.\n\n"
    "Tone definitions:\n"
    "- formal: professional register, complete sentences, polite and structured\n"
    "- informal: casual tone, contractions, colloquialisms, friendly phrasing\n"
    "- urgent: time pressure, urgency markers (ASAP, immediately), high stakes\n"
    "- technical: domain-specific jargon, error codes, API/system terminology, engineering context"
)


def generate_examples_v2(label: str, n: int, llm: ChatOpenAI) -> list[str]:
    """Extended generator with four-label support."""
    prompt = (
        f"Generate {n} diverse customer service messages that have a clearly '{label}' tone.\n"
        "Requirements:\n"
        "- Each message should be on its own line\n"
        "- Vary the industry, product type, and phrasing\n"
        "- Include a mix of short (1 sentence) and medium (2-3 sentence) messages\n"
        "- Make each message distinctly different from the others\n"
        f"- Number each message: 1. ... 2. ...\n\n"
        f"Generate {n} messages now:"
    )
    response = llm.invoke([
        SystemMessage(content=GENERATOR_SYSTEM_V2),
        HumanMessage(content=prompt),
    ])
    raw = response.content.strip()
    examples = []
    for line in raw.split("\n"):
        line = line.strip()
        if not line:
            continue
        for i in range(1, n + 10):
            if line.startswith(f"{i}. "):
                line = line[len(f"{i}. "):].strip()
                break
        if len(line) > 10:
            examples.append(line)
    return examples[:n]


gen_llm_v2 = ChatOpenAI(model="gpt-4o-mini", temperature=0.9)
technical_examples = generate_examples_v2("technical", n=5, llm=gen_llm_v2)

print("Generated technical label examples:")
for i, msg in enumerate(technical_examples, 1):
    print(f"  {i}. {msg}")

In [ ]:
# ===== ANSWER KEY — Exercise 2: Tune deduplication threshold =====

# Reuse the 15-example batch with embeddings already computed (raw_examples)
thresholds_to_test = [0.85, 0.90, 0.95, 0.99]

print(f"{'Threshold':<12} {'Kept':<8} {'Dropped':<10} {'Dedup rate'}")
print("-" * 45)

for threshold in thresholds_to_test:
    kept_examples, n_drop = deduplicate_by_embedding(raw_examples, threshold=threshold)
    rate = n_drop / len(raw_examples) * 100 if raw_examples else 0
    print(f"  {threshold:<10.2f} {len(kept_examples):<8} {n_drop:<10} {rate:.0f}%")

print()
print("Observations:")
print("  0.85: aggressive — removes stylistically similar but semantically distinct examples")
print("  0.90: moderate — removes paraphrases and some topic-similar examples")
print("  0.95: conservative (default) — removes near-identical paraphrases only")
print("  0.99: minimal — removes only exact or near-exact duplicates")
print()
print("Start inspecting dropped pairs when rate jumps — that's where you're over-filtering.")

In [ ]:
# ===== ANSWER KEY — Exercise 3: Stratified train/val split =====

import random
from collections import defaultdict


def split_train_val(
    examples: list[GeneratedExample],
    val_fraction: float = 0.2,
    seed: int = 42,
) -> tuple[list[GeneratedExample], list[GeneratedExample]]:
    """
    Stratified train/val split — preserves label proportions in both splits.

    Args:
        examples: list of GeneratedExample (with is_valid=True examples only, typically)
        val_fraction: fraction of each label to put in val set
        seed: random seed for reproducibility

    Returns:
        (train_examples, val_examples)
    """
    rng = random.Random(seed)

    # Group by label
    by_label: dict[str, list[GeneratedExample]] = defaultdict(list)
    for ex in examples:
        by_label[ex.label].append(ex)

    train: list[GeneratedExample] = []
    val: list[GeneratedExample] = []

    for label, label_examples in by_label.items():
        shuffled = label_examples[:]
        rng.shuffle(shuffled)
        n_val = max(1, int(len(shuffled) * val_fraction))
        val.extend(shuffled[:n_val])
        train.extend(shuffled[n_val:])

    return train, val


# Use the judged examples from the pipeline run above
valid_examples = [ex for ex in judged if ex.is_valid]

train_set, val_set = split_train_val(valid_examples, val_fraction=0.2)

print(f"Valid examples:  {len(valid_examples)}")
print(f"Train set:       {len(train_set)}")
print(f"Val set:         {len(val_set)}")

# Show label distribution in each split
def label_dist(examples: list[GeneratedExample]) -> dict[str, int]:
    counts: dict[str, int] = defaultdict(int)
    for ex in examples:
        counts[ex.label] += 1
    return dict(counts)

print(f"\nTrain label distribution: {label_dist(train_set)}")
print(f"Val   label distribution: {label_dist(val_set)}")

# Export both splits
train_path = "/tmp/train.jsonl"
val_path = "/tmp/val.jsonl"

train_count = export_to_jsonl(train_set, train_path)
val_count = export_to_jsonl(val_set, val_path)

print(f"\nExported {train_count} train examples → {train_path}")
print(f"Exported {val_count}  val   examples → {val_path}")

## Workshop Complete

You have built a complete synthetic training data pipeline:

- **Stage 1 — Generator**: GPT-4o with a diversity-optimised prompt generates labeled examples across multiple industries, lengths, and phrasings. High temperature (0.9) maximises variety.
- **Stage 2 — Deduplication**: `text-embedding-3-small` turns each message into a 1536-d vector. Cosine similarity above 0.95 identifies paraphrases and drops them. Prevents mode-collapse artifacts from degrading fine-tune quality.
- **Stage 3 — Judge**: A second LLM call at temperature=0 validates label correctness and flags ambiguous examples. JSON-structured output makes parsing reliable. Ambiguous examples are excluded from training.
- **Export**: JSONL in OpenAI fine-tuning chat format — ready to upload directly to the fine-tuning API.

**The key insight**: the generator-deduplicator-judge triad separates three concerns — diversity, uniqueness, and accuracy — into independent, tunable stages. You can swap any stage independently: use a different embedding model, raise the dedup threshold, replace the judge with a rule-based validator.

For 300 high-quality examples, the entire pipeline costs under $0.03. Fine-tuning `gpt-4o-mini` on this dataset costs under $0.30. The resulting model classifies customer service tone accurately and cheaply — indefinitely.

---

Next: **example 113** — continuing the fine-tuning series.

---

### Further reading

- [arxiv:2212.10560](https://arxiv.org/abs/2212.10560) — Wang et al. Self-Instruct (the paper that started the synthetic data flywheel)
- [arxiv:2304.12244](https://arxiv.org/abs/2304.12244) — Xu et al. WizardLM (evolved Self-Instruct with complexity-increasing rewriting)
- [arxiv:2107.06499](https://arxiv.org/abs/2107.06499) — Lee et al. Deduplicating Training Data (why dedup improves LM quality)
- [OpenAI Fine-tuning guide](https://platform.openai.com/docs/guides/fine-tuning) — format, training API, hyperparameters
- [OpenAI text-embedding-3-small](https://platform.openai.com/docs/guides/embeddings) — embedding model used in Stage 2